In [1]:
import pandas as pd
import numpy as np

def load_data(file_path):
    """
    Load data from a CSV file into a pandas DataFrame.

    Parameters:
    file_path (str): The path to the CSV file.

    Returns:
    pd.DataFrame: A DataFrame containing the loaded data.
    """
    try:
        data = pd.read_csv(file_path, sep=';')
        return data
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

In [2]:
dataset = load_data('../dataset/dataset.csv') # Memuat data mentah dari file CSV ke dalam variabel dataset

print("Data loaded successfully. Here are the first few rows:")
print(dataset.head()) # Menampilkan 5 baris teratas untuk mengecek apakah data berhasil dimuat

Data loaded successfully. Here are the first few rows:
       Tanggal         Kategori Barang Kode Barang        Nama Barang  \
0  01 Jan 2024  Barang Semi FG (WIP-2)   FGS-00001  Ayam Kebuli (0.9)   
1  01 Jan 2024  Barang Semi FG (WIP-2)   FGS-00002     Kambing Kebuli   
2  01 Jan 2024  Barang Semi FG (WIP-2)   FGS-00003    Iga Sapi Kebuli   
3  01 Jan 2024  Barang Semi FG (WIP-2)   FGS-00004        Nasi Kebuli   
4  01 Jan 2024  Barang Semi FG (WIP-2)   FGS-00005        Sambal - FG   

                             Nama Cabang  Satuan  Kuantitas  
0  KY001 - Kebuli Yaman Kutabumi (Pusat)  Potong        235  
1  KY001 - Kebuli Yaman Kutabumi (Pusat)   Porsi          3  
2  KY001 - Kebuli Yaman Kutabumi (Pusat)   Porsi          7  
3  KY001 - Kebuli Yaman Kutabumi (Pusat)   Porsi        256  
4  KY001 - Kebuli Yaman Kutabumi (Pusat)   Porsi        263  


In [3]:
# dataset.to_excel('../dataset/excel/exported_clean_data.xlsx', index=False)

# print("Export complete!")

In [4]:
import sys
import os

# Add the parent directory to the system path
sys.path.append(os.path.abspath('..'))

In [5]:
import normalize_items # Mengimpor modul kustom untuk menormalkan kode barang
import build_panel # Mengimpor modul untuk membuat panel harian
import calendar_features # Mengimpor modul untuk merekayasa fitur kalender/hari libur
import prepare_forecast_data # Mengimpor modul untuk persiapan akhir data forecast
import outlet_features # Mengimpor modul untuk fitur lokasi & kanal online outlet

outlets_df = outlet_features.load_outlets() # Memuat data master outlet (kota, kanal online) dari dataset/outlets.csv
overrides_df = outlet_features.load_overrides() # Memuat pemetaan koreksi nama cabang -> outlet


### Compare existing outlets

In [6]:
# Compare every branch name in the raw dataset against dataset/outlets.csv (+ manual overrides),
# before any cleaning/normalization, to see which branches are recognized outlets and which are not
raw_branches = sorted(dataset["Nama Cabang"].unique())
matched_branches = {
    b for b in raw_branches
    if outlet_features.match_branch_to_outlet(b, outlets_df, overrides_df)[0] is not None
}
unmatched_branches = sorted(set(raw_branches) - matched_branches)

print(f"{len(matched_branches)} of {len(raw_branches)} branch(es) match an outlet in outlets.csv")
if unmatched_branches:
    print(f"{len(unmatched_branches)} branch(es) have no outlets.csv match:")
    for b in unmatched_branches:
        print(f"  - {b}")
else:
    print("Every branch in the raw dataset matches an outlet.")


56 of 67 branch(es) match an outlet in outlets.csv


11 branch(es) have no outlets.csv match:
  - KY020 - Kebuli Yaman Tambun
  - KY028 - Kebuli Yaman Condet
  - KY035 - Kebuli Yaman Antapani
  - KY046 - Kebuli Yaman Aryana Karawaci
  - KY047 - Kebuli Yaman Ciomas
  - KY052 - Kebuli Yaman Bantarjati Bogor
  - KY055 - Kebuli Yaman Ciputat Timur
  - KY059 - Kebuli Yaman Dukuh Zamrud
  - KY071 - Kebuli Yaman Citayam
  - KY072 - Kebuli Yaman Bintara
  - Kebab Saudagar - Kutabumi


### Check data and missing data

In [7]:
# Check for missing values in every column
print(dataset.isnull().sum()) # Menghitung jumlah nilai yang kosong (null/NaN) pada setiap kolom

Tanggal            0
Kategori Barang    0
Kode Barang        0
Nama Barang        0
Nama Cabang        0
Satuan             0
Kuantitas          0
dtype: int64


In [8]:
# Count the number of unique entries per column
print(dataset.nunique()) # Menghitung jumlah variasi nilai unik untuk masing-masing kolom (misal: ada berapa banyak cabang unik?)

Tanggal            731
Kategori Barang     12
Kode Barang        109
Nama Barang        112
Nama Cabang         67
Satuan              12
Kuantitas          819
dtype: int64


In [9]:
# List the columns you want to inspect
columns_to_check = ['Kategori Barang', 'Kode Barang', 'Nama Barang', 'Nama Cabang', 'Satuan'] # Mendefinisikan kolom-kolom kategorikal yang isinya ingin kita intip

# Loop through each column and print its unique contents
for col in columns_to_check: # Melakukan perulangan untuk setiap kolom di atas
    print(f"--- Data available in '{col}' ---")
    print(dataset[col].unique()) # Mencetak semua nilai unik/kategori yang ada di dalam kolom tersebut
    print("-" * 50)

--- Data available in 'Kategori Barang' ---


['Barang Semi FG (WIP-2)' 'Minuman' 'Packaging' 'Bahan Baku (RM)'
 'Barang Dalam Process (WIP-1)' 'Snack' 'Barang Umum' 'Perlengkapan Resto'
 'Minuman - FG' 'Barang Jadi (FG)' 'Snack (FG)' 'Tambahan']
--------------------------------------------------
--- Data available in 'Kode Barang' ---
['FGS-00001' 'FGS-00002' 'FGS-00003' 'FGS-00004' 'FGS-00005' 'FGS-00009'
 'FGS-00013' 'FGS-00015' 'FGS-00017' 'PCG-00001' 'PCG-00003' 'PCG-00004'
 'PCG-00005' 'PCG-00006' 'PCG-00007' 'PCG-00008' 'PCG-00009' 'PCG-00010'
 'PCG-00011' 'PCG-00012' 'PCG-00013' 'PCG-00024' 'PCG-00030' 'RMT-00017'
 'FGS-00039' 'FGS-00040' 'FGS-00041' 'PCG-00021' 'FGS-00014' 'FGS-00008'
 'FGS-00007' 'FGS-00035' 'FGS-00037' 'WIP.00009' 'FGS-00047' 'FGS-00038'
 'FGS-00056' 'FGS-00006' 'FGS-00012' 'PCG-00036' 'FGS-00042' 'FGS-00043'
 'FGS-00053' 'FGS-00048' 'FGS.00047' 'FGS-00046' 'FGS-00055' 'FGS-00044'
 'FGS-00045' 'Kalender' 'FGS-00018' 'PCG-00002' 'PCG-00027' 'PCG-00028'
 'PCG-00022' 'FGS.00049' 'SPR.00004' 'FGS-00032' 'WI

In [10]:
# Shows all 67 branches, sorted from most frequent to least frequent
with pd.option_context('display.max_rows', None): # Mengubah pengaturan tampilan pandas sementara agar menampilkan semua baris (tanpa dipotong)
    # Use display() and .to_frame() to create a neat, scrollable table
    display(dataset['Nama Cabang'].value_counts().to_frame()) # Menghitung jumlah transaksi per cabang lalu menampilkannya dalam bentuk tabel yang rapi

print(f"Which outlets are available: {len(matched_branches)} branch(es) with a recognized outlet, {len(unmatched_branches)} without (see compare step above)")
print(f"Which items are available: {dataset['Kode Barang'].nunique()} unique item code(s) across {dataset['Nama Barang'].nunique()} unique item name(s)")


,count
Nama Cabang,
KY001 - Kebuli Yaman Kutabumi (Pusat),20032
KY042 - Kebuli Yaman Batavia,17118
KY041 - Kebuli Yaman Sepatan,16695
KY007 - Kebuli Yaman Cibubur,16331
KY003 - Kebuli Yaman Serang,16231
KY016 - Kebuli Yaman Jatimakmur,16199
KY040 - Kebuli Yaman Anyer,15947
KY006 - Kebuli Yaman Depok Sentosa,15944
KY002 - Kebuli Yaman Cilegon,15827


Which outlets are available: 56 branch(es) with a recognized outlet, 11 without (see compare step above)


Which items are available: 109 unique item code(s) across 112 unique item name(s)


### Check duplicated data

In [11]:
total_duplicates = dataset.duplicated().sum() # Menghitung total baris yang menduplikat baris lain secara persis (identik di semua kolom)
print(f"Total duplicate rows: {total_duplicates}")
print("-" * 40)

# (keep=False ensures we see the original row AND its duplicates)
duplicates = dataset[dataset.duplicated(keep=False)] # Menyimpan semua baris yang duplikat agar bisa diinspeksi

if total_duplicates > 0:
    print("Here is a peek at the duplicate rows:")
    print(duplicates.sort_values(by=['Tanggal', 'Kode Barang', 'Nama Cabang']).head(10)) # Menampilkan 10 baris duplikat teratas yang diurutkan
else:
    print("Your dataset is clean! No identical duplicate rows found.")

Total duplicate rows: 0
----------------------------------------


Your dataset is clean! No identical duplicate rows found.


In [12]:
business_duplicates = dataset.duplicated(subset=['Tanggal', 'Kode Barang', 'Nama Cabang']) # Mengecek apakah ada transaksi dengan barang dan cabang yang sama di tanggal yang sama (duplikat secara bisnis)
print(f"Duplicates based on specific columns: {business_duplicates.sum()}") # Menampilkan jumlah duplikat bisnis

Duplicates based on specific columns: 0


### Normalize Nama Barang and Kode Barang

Also folds in branch-name canonicalization (matching/filtering/renaming `Nama Cabang` against `outlets.csv`) since `build_panel` needs both item and branch cleanup done together.

In [13]:
normalized = normalize_items.load_and_normalize() # Membaca raw data dan membersihkan penulisan/typo pada Kode Barang & mengagregasi ulang
normalized = outlet_features.filter_matched_branches(normalized, outlets_df, overrides_df) # Membuang baris untuk cabang yang tidak ada padanannya di outlets.csv (cabang sudah tidak beroperasi)
normalized = outlet_features.canonicalize_branch_names(normalized, outlets_df, overrides_df) # Menyeragamkan nama cabang legacy/duplikat (mis. "TOD M1 Bandara") ke nama outlet kanoniknya
normalized = normalize_items.reaggregate_daily(normalized) # Menggabungkan ulang baris yang kini bertabrakan (barang, tanggal, cabang) akibat penyeragaman nama cabang di atas
print(f"{len(normalized)} rows after item-code normalization, reaggregation, unmatched-branch filtering, and branch-name canonicalization")


624290 rows after item-code normalization, reaggregation, unmatched-branch filtering, and branch-name canonicalization


In [14]:
# normalized.to_excel('../dataset/excel/normalized_data.xlsx', index=False) # Mengekspor data yang sudah bersih ke excel untuk keperluan inspeksi manual

### Check completeness data

In [15]:
start_date = normalized['Tanggal'].min() # Mencari tanggal paling awal di data yang sudah dinormalisasi
end_date = normalized['Tanggal'].max() # Mencari tanggal paling akhir di data yang sudah dinormalisasi

print(f"Checking data from {start_date.strftime('%d %b %Y')} to {end_date.strftime('%d %b %Y')}") # Menampilkan rentang tanggal data
print("-" * 40)

expected_dates = pd.date_range(start=start_date, end=end_date) # Membuat daftar semua tanggal yang seharusnya ada antara start_date dan end_date

actual_dates = normalized['Tanggal'].dropna().unique() # Mengambil semua tanggal unik yang benar-benar ada di data

missing_dates = expected_dates.difference(actual_dates) # Mencari tanggal yang ada di expected_dates tapi tidak ada di actual_dates (tanggal bolong)

if missing_dates.empty:
    print("Your daily data is complete! No missing dates.") # Jika tidak ada tanggal yang bolong
else:
    print(f"You are missing data for {len(missing_dates)} days.") # Jika ada tanggal yang bolong
    print("Missing dates:")
    print(missing_dates.strftime('%d %b %Y').tolist())


Checking data from 01 Jan 2024 to 31 Dec 2025
----------------------------------------
Your daily data is complete! No missing dates.


In [16]:
panel = build_panel.build_dense_panel(normalized) # Membangun panel harian padat (mengisi tanggal yang bolong dengan Kuantitas=0 untuk tiap Cabang-Barang)
print(f"{len(panel)} rows after dense daily zero-fill ({panel[build_panel.PAIR_COLS].drop_duplicates().shape[0]} item-branch pairs)")

1357631 rows after dense daily zero-fill (3241 item-branch pairs)


In [17]:
panel = build_panel.filter_min_history(panel) # Membuang cabang-barang yang riwayat transaksinya kurang dari 60 hari (data terlalu sedikit untuk dilatih)
print(f"{len(panel)} rows after minimum-history filter ({panel[build_panel.PAIR_COLS].drop_duplicates().shape[0]} item-branch pairs)")

1341346 rows after minimum-history filter (2636 item-branch pairs)


In [18]:
# List the columns you want to inspect
columns_to_check = ['Kategori Barang', 'Kode Barang', 'Nama Barang', 'Nama Cabang'] # Mendefinisikan kolom-kolom kategorikal yang isinya ingin kita intip

# Loop through each column and print its unique contents
for col in columns_to_check: # Melakukan perulangan untuk setiap kolom di atas
    print(f"--- Data available in '{col}' ---")
    print(panel[col].unique()) # Mencetak semua nilai unik/kategori yang ada di dalam kolom tersebut
    print("Total data: ", panel[col].nunique())
    print("-" * 50)

--- Data available in 'Kategori Barang' ---
['Barang Semi FG (WIP-2)' 'Barang Jadi (FG)' 'Minuman' 'Minuman - FG'
 'Snack' 'Snack (FG)' 'Barang Umum' 'Packaging' 'Bahan Baku (RM)'
 'Barang Dalam Process (WIP-1)']
Total data:  10
--------------------------------------------------
--- Data available in 'Kode Barang' ---


['FGS-00001' 'FGS-00002' 'FGS-00003' 'FGS-00004' 'FGS-00005' 'FGS-00006'
 'FGS-00007' 'FGS-00008' 'FGS-00009' 'FGS-00011' 'FGS-00012' 'FGS-00013'
 'FGS-00014' 'FGS-00015' 'FGS-00017' 'FGS-00018' 'FGS-00034' 'FGS-00035'
 'FGS-00037' 'FGS-00038' 'FGS-00039' 'FGS-00040' 'FGS-00041' 'FGS-00042'
 'FGS-00043' 'FGS-00044' 'FGS-00045' 'FGS-00046' 'FGS-00047' 'FGS-00048'
 'FGS-00049' 'FGS-00050' 'FGS-00051' 'FGS-00052' 'FGS-00053' 'FGS-00054'
 'FGS-00055' 'FGS-00056' 'FGS-00065' 'FGS-00068' 'FGS-00069' 'FGS-00070'
 'FGS-00071' 'FGS.00048' 'FGS.00053' 'FGS.00055' 'FGS.00056' 'Kalender'
 'PCG-00001' 'PCG-00002' 'PCG-00003' 'PCG-00004' 'PCG-00005' 'PCG-00006'
 'PCG-00007' 'PCG-00008' 'PCG-00009' 'PCG-00010' 'PCG-00011' 'PCG-00012'
 'PCG-00013' 'PCG-00021' 'PCG-00024' 'PCG-00027' 'PCG-00028' 'PCG-00030'
 'PCG-00036' 'PCG-00038' 'PCG-00044' 'RMT-00017' 'WIP-00009']
Total data:  71
--------------------------------------------------
--- Data available in 'Nama Barang' ---


['Ayam Kebuli (0.9)' 'Kambing Kebuli' 'Iga Sapi Kebuli' 'Nasi Kebuli'
 'Sambal - FG' 'Club Mineral 330 ml' '7dates Jus Kurma'
 '7dates Susu Kurma' 'Dahagaku Lemon Sereh 250ml'
 'Saus Extra Delmonte @8gr' 'Samosa Beef Original (RM)'
 'Samosa Beef Spicy (RM)' 'Club Mineral 600 ml (Menu Pakai Kode CM-600)'
 'Ichi Ocha Melati 350 ml' 'Susu Almond - FG'
 'Kambing Kebuli Aqiqah Betina - FG' 'Kambing Kebuli Aqiqah Jantan - FG'
 'Gula Asam 250ml' 'Kunyit Asam 250ml' 'Meet Jelly Coklat'
 'Meet Jelly Mangga' 'Meet Jelly Strawberry' 'Meet Jelly Leci'
 'Meet Jelly Kopi Susu' 'Meet Jelly Teh Tarik' 'Kentang Mustofa Rabeg'
 'Kentang Mustofa Balado' 'Kentang Mustofa Keju Asin'
 'Kentang Mustofa Rumput Laut' 'Kentang Mustofa Mie Goreng'
 'Iga Dino - FG' 'AirAlam 600 ml (Menu Pakai Kode AA-600)'
 'Iga Jontor - FG' 'Iga Sapi Original - FG' 'Ayam Kebuli (0.6)'
 'India Salaam Basmati Rice @1kg' 'Chicken Kebab' 'Beef Kebab'
 'Saus Tomat Delmonte @8gr' 'Ayam Crispy Spicy - FG' 'Cendol - FG'
 'Santan Cendol 

['KY001 - Kebuli Yaman Kutabumi (Pusat)' 'KY002 - Kebuli Yaman Cilegon'
 'KY003 - Kebuli Yaman Serang' 'KY004 - Kebuli Yaman Depok Sawangan'
 'KY005 - Kebuli Yaman Pandeglang' 'KY006 - Kebuli Yaman Depok Sentosa'
 'KY007 - Kebuli Yaman Cibubur' 'KY008 - Kebuli Yaman Depok Kelapa Dua'
 'KY009 - Kebuli Yaman Cipinang' 'KY010 - Kebuli Yaman Bogor Ciampea'
 'KY011 - Kebuli Yaman Bekasi Galaxy' 'KY012 - Kebuli Yaman Cibinong'
 'KY013 - Kebuli Yaman Cileunyi' 'KY014 - Kebuli Yaman Ciparay'
 'KY016 - Kebuli Yaman Jatimakmur' 'KY019 - Kebuli Yaman Ciledug'
 'KY021 - Kebuli Yaman Cikeas' 'KY022 - Kebuli Yaman Citra Raya'
 'KY023 - Kebuli Yaman Pondok Kelapa' 'KY026 - Kebuli Yaman Poris'
 'KY027 - Kebuli Yaman Kaliabang' 'KY029 - Kebuli Yaman Cinere'
 'KY031 - Kebuli Yaman Perum' 'KY032 - Kebuli Yaman Kelapa Dua Karawaci'
 'KY033 - Kebuli Yaman Jatiasih' 'KY036 - Kebuli Yaman Kota Harapan Indah'
 'KY037 - Kebuli Yaman Hankam' 'KY038 - Kebuli Yaman Talaga Bestari'
 'KY040 - Kebuli Yaman Anyer' 'K

In [19]:
featured = prepare_forecast_data.add_targets(panel) # Membuat kolom target H+1 s/d H+7 (label untuk diprediksi)
featured = prepare_forecast_data.add_lag_features(featured) # Membuat fitur lag (data historis h-1, h-2, dll)
featured = prepare_forecast_data.add_rolling_features(featured) # Membuat fitur rolling statistik (rata-rata & deviasi dalam 7, 14, 28 hari terakhir)
featured = calendar_features.add_calendar_features(featured) # Menambahkan fitur kalender (hari libur nasional, akhir pekan, Ramadhan, dll)
branch_stats = prepare_forecast_data.compute_branch_stats(featured) # Menghitung rata-rata & variansi spesifik untuk tiap cabang
featured = prepare_forecast_data.apply_branch_stats(featured, branch_stats) # Menempelkan nilai statistik cabang tersebut kembali ke data utama
featured = prepare_forecast_data.add_branch_age_days(featured) # Menghitung umur cabang (sudah berapa hari cabang tersebut beroperasi sejak transaksi pertamanya)
featured = prepare_forecast_data.apply_outlet_features(featured, outlets_df, overrides_df) # Menempelkan kota, has_shopee/gofood/grabfood, dan can_order_online per cabang
print(f"{len(featured)} rows, {len(featured.columns)} columns after feature engineering")

1341346 rows, 55 columns after feature engineering


In [20]:
# Row-count sanity check
expected_rows = panel[build_panel.PAIR_COLS].drop_duplicates().shape[0] # Memastikan jumlah unik kombinasi cabang-barang tidak berubah
assert len(featured) == len(panel), "feature engineering must not change row count" # Memastikan tidak ada baris yang bertambah/berkurang setelah proses *feature engineering*
print("Row count sanity check passed.")

Row count sanity check passed.


In [21]:
# No duplicate (pair, date) rows
dup_count = featured.duplicated(subset=build_panel.PAIR_COLS + ["Tanggal"]).sum() # Menghitung baris ganda berdasarkan cabang, barang, dan tanggal
assert dup_count == 0, f"found {dup_count} duplicate (pair, date) rows" # Validasi akhir untuk memastikan setiap tanggal hanya punya 1 baris per barang per cabang
print("No duplicate (pair, date) rows.")

No duplicate (pair, date) rows.


In [22]:
# No negative quantities
assert (featured["Kuantitas"] >= 0).all(), "found negative Kuantitas values" # Memastikan tidak ada Kuantitas penjualan yang bernilai minus/negatif
print("No negative Kuantitas values.")

No negative Kuantitas values.


In [23]:
# Ramadan/Eid flags spot-check against verified dates
ramadan_2024_start = featured[featured["Tanggal"] == "2024-03-11"] # Mengecek baris data pada tanggal 11 Maret 2024
assert ramadan_2024_start["is_ramadan"].all(), "2024-03-11 should be flagged as Ramadan" # Memastikan kolom 'is_ramadan' bernilai True untuk tanggal 11 Maret 2024
eid_fitr_2024 = featured[featured["Tanggal"] == "2024-04-10"] # Mengecek baris data pada tanggal 10 April 2024
assert eid_fitr_2024["is_eid_al_fitr"].all(), "2024-04-10 should be flagged as Eid al-Fitr" # Memastikan kolom 'is_eid_al_fitr' bernilai True untuk tanggal tersebut
print("Ramadan/Eid flag spot-check passed.")

Ramadan/Eid flag spot-check passed.


In [24]:
# Leakage check: a sampled row's lag/rolling features only reference strictly earlier dates
sample_pair = featured[build_panel.PAIR_COLS].drop_duplicates().iloc[0] # Mengambil satu contoh acak pasangan cabang & barang
sample_series = featured[
    (featured["Kode Barang"] == sample_pair["Kode Barang"])
    & (featured["Nama Cabang"] == sample_pair["Nama Cabang"])
].sort_values("Tanggal").reset_index(drop=True) # Mengambil semua riwayat data untuk sampel tersebut dan mengurutkannya berdasarkan tanggal

if len(sample_series) > 28:
    row = sample_series.iloc[28]  # far enough in to have full lag history (mengambil hari ke-29 agar lag 28 hari sudah terisi)
    assert row["lag_1"] == sample_series.iloc[27]["Kuantitas"], "lag_1 must equal the prior day's Kuantitas" # Memastikan fitur lag_1 nilainya memang sama dengan kuantitas h-1
    assert row["roll_mean_7"] == sample_series.iloc[21:28]["Kuantitas"].mean(), "roll_mean_7 must exclude today's own Kuantitas" # Memastikan rata-rata 7 hari tidak mengikutsertakan kuantitas hari-H (mencegah kebocoran data masa depan)
    print("Leakage spot-check passed: lag_1 and roll_mean_7 reference only strictly earlier dates.")
else:
    print("Sampled pair too short for a 28-day leakage check; pick a longer-history pair if this happens.")

Leakage spot-check passed: lag_1 and roll_mean_7 reference only strictly earlier dates.


In [25]:
# Outlet-match QA: no branch should remain "Unknown" now that unmatched branches are dropped upstream
unknown_branches = sorted(featured.loc[featured["kota"] == "Unknown", "Nama Cabang"].unique()) # Mengambil daftar cabang yang tidak berhasil dicocokkan dengan outlet manapun (seharusnya kosong)
assert not unknown_branches, f"unexpected unmatched branches survived filtering: {unknown_branches}" # Jika ada cabang yang lolos filter tapi tetap Unknown, berarti ada bug pada tahap filtering
print("No branches remain with kota == 'Unknown' (unmatched branches were dropped upstream).")

No branches remain with kota == 'Unknown' (unmatched branches were dropped upstream).


In [26]:
# Outlet-match QA: every branch maps to exactly one outlet (no fan-out)
branch_outlet_counts = featured.groupby("Nama Cabang")["kota"].nunique() # Menghitung berapa banyak nilai kota berbeda per cabang
fanned_out = branch_outlet_counts[branch_outlet_counts > 1] # Cabang yang punya lebih dari satu nilai kota berarti ada penggandaan (fan-out) hasil join
assert fanned_out.empty, f"branches with more than one outlet match: {list(fanned_out.index)}" # Memastikan tidak ada cabang yang match ke lebih dari satu baris outlet
print("Every branch maps to exactly one outlet (no fan-out).")

Every branch maps to exactly one outlet (no fan-out).


In [27]:
# can_order_online distribution sanity check
branch_level = featured.drop_duplicates(subset=["Nama Cabang"])[["Nama Cabang", "can_order_online"]] # Mengambil satu baris per cabang (fitur ini statis per cabang, bukan per hari)
print(branch_level["can_order_online"].value_counts(dropna=False)) # Menampilkan jumlah cabang True/False/NaN (tidak diketahui)

can_order_online
True     50
False     2
Name: count, dtype: int64


In [28]:
featured.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1341346 entries, 0 to 1341345
Data columns (total 55 columns):
 #   Column                       Non-Null Count    Dtype         
---  ------                       --------------    -----         
 0   Kode Barang                  1341346 non-null  object        
 1   Nama Cabang                  1341346 non-null  object        
 2   Tanggal                      1341346 non-null  datetime64[ns]
 3   Kuantitas                    1341346 non-null  float64       
 4   Kategori Barang              1341346 non-null  object        
 5   Nama Barang                  1341346 non-null  object        
 6   target_h1                    1338710 non-null  float64       
 7   target_h2                    1336074 non-null  float64       
 8   target_h3                    1333438 non-null  float64       
 9   target_h4                    1330802 non-null  float64       
 10  target_h5                    1328166 non-null  float64       
 11  target_h6  

In [29]:
# List the columns you want to inspect
columns_to_check = ['Kategori Barang', 'Kode Barang', 'Nama Barang', 'Nama Cabang'] # Mendefinisikan kolom-kolom kategorikal yang isinya ingin kita intip

# Loop through each column and print its unique contents
for col in columns_to_check: # Melakukan perulangan untuk setiap kolom di atas
    print(f"--- Data available in '{col}' ---")
    print(featured[col].unique()) # Mencetak semua nilai unik/kategori yang ada di dalam kolom tersebut
    print("Total data: ", featured[col].nunique())
    print("-" * 50)

--- Data available in 'Kategori Barang' ---
['Barang Semi FG (WIP-2)' 'Barang Jadi (FG)' 'Minuman' 'Minuman - FG'
 'Snack' 'Snack (FG)' 'Barang Umum' 'Packaging' 'Bahan Baku (RM)'
 'Barang Dalam Process (WIP-1)']
Total data:  10
--------------------------------------------------
--- Data available in 'Kode Barang' ---
['FGS-00001' 'FGS-00002' 'FGS-00003' 'FGS-00004' 'FGS-00005' 'FGS-00006'
 'FGS-00007' 'FGS-00008' 'FGS-00009' 'FGS-00011' 'FGS-00012' 'FGS-00013'
 'FGS-00014' 'FGS-00015' 'FGS-00017' 'FGS-00018' 'FGS-00034' 'FGS-00035'
 'FGS-00037' 'FGS-00038' 'FGS-00039' 'FGS-00040' 'FGS-00041' 'FGS-00042'
 'FGS-00043' 'FGS-00044' 'FGS-00045' 'FGS-00046' 'FGS-00047' 'FGS-00048'
 'FGS-00049' 'FGS-00050' 'FGS-00051' 'FGS-00052' 'FGS-00053' 'FGS-00054'
 'FGS-00055' 'FGS-00056' 'FGS-00065' 'FGS-00068' 'FGS-00069' 'FGS-00070'
 'FGS-00071' 'FGS.00048' 'FGS.00053' 'FGS.00055' 'FGS.00056' 'Kalender'
 'PCG-00001' 'PCG-00002' 'PCG-00003' 'PCG-00004' 'PCG-00005' 'PCG-00006'
 'PCG-00007' 'PCG-00008'

['Ayam Kebuli (0.9)' 'Kambing Kebuli' 'Iga Sapi Kebuli' 'Nasi Kebuli'
 'Sambal - FG' 'Club Mineral 330 ml' '7dates Jus Kurma'
 '7dates Susu Kurma' 'Dahagaku Lemon Sereh 250ml'
 'Saus Extra Delmonte @8gr' 'Samosa Beef Original (RM)'
 'Samosa Beef Spicy (RM)' 'Club Mineral 600 ml (Menu Pakai Kode CM-600)'
 'Ichi Ocha Melati 350 ml' 'Susu Almond - FG'
 'Kambing Kebuli Aqiqah Betina - FG' 'Kambing Kebuli Aqiqah Jantan - FG'
 'Gula Asam 250ml' 'Kunyit Asam 250ml' 'Meet Jelly Coklat'
 'Meet Jelly Mangga' 'Meet Jelly Strawberry' 'Meet Jelly Leci'
 'Meet Jelly Kopi Susu' 'Meet Jelly Teh Tarik' 'Kentang Mustofa Rabeg'
 'Kentang Mustofa Balado' 'Kentang Mustofa Keju Asin'
 'Kentang Mustofa Rumput Laut' 'Kentang Mustofa Mie Goreng'
 'Iga Dino - FG' 'AirAlam 600 ml (Menu Pakai Kode AA-600)'
 'Iga Jontor - FG' 'Iga Sapi Original - FG' 'Ayam Kebuli (0.6)'
 'India Salaam Basmati Rice @1kg' 'Chicken Kebab' 'Beef Kebab'
 'Saus Tomat Delmonte @8gr' 'Ayam Crispy Spicy - FG' 'Cendol - FG'
 'Santan Cendol 

Total data:  52
--------------------------------------------------


### Check outlet date completeness (2024-01-01 to 2025-12-31)

In [30]:
# Full expected daily range for every outlet, regardless of the data's own min/max
full_range = pd.date_range("2024-01-01", "2025-12-31", freq="D")  # Rentang tanggal lengkap yang diharapkan untuk setiap outlet, 1 Jan 2024 s/d 31 Des 2025

# Dates actually present per outlet (any item), so a branch only counts as "missing" a day if it has zero rows at all that day
outlet_dates = featured.groupby("Nama Cabang")["Tanggal"].unique()  # Mengambil semua tanggal unik yang benar-benar ada per outlet (gabungan semua barang)

missing_by_outlet = {}
for outlet, dates in outlet_dates.items():  # Memeriksa satu per satu outlet
    missing = full_range.difference(pd.to_datetime(dates))  # Mencari tanggal yang seharusnya ada tapi tidak ditemukan untuk outlet ini
    if not missing.empty:
        missing_by_outlet[outlet] = missing

if not missing_by_outlet:
    print("All outlets have complete daily data from 2024-01-01 to 2025-12-31.")  # Semua outlet punya data harian lengkap, tidak ada tanggal bolong
else:
    print(f"{len(missing_by_outlet)} outlet(s) have missing dates in 2024-01-01 to 2025-12-31:")  # Menampilkan jumlah outlet yang datanya tidak lengkap
    for outlet, missing in sorted(missing_by_outlet.items(), key=lambda kv: -len(kv[1])):  # Diurutkan dari outlet dengan tanggal bolong terbanyak
        preview = missing.strftime("%d %b %Y").tolist()
        preview_str = ", ".join(preview[:5]) + (", ..." if len(preview) > 5 else "")
        print(f"  - {outlet}: {len(missing)} missing day(s) (e.g. {preview_str})")  # Detail per outlet: jumlah & contoh tanggal yang hilang

16 outlet(s) have missing dates in 2024-01-01 to 2025-12-31:
  - KY066 - Kebuli Yaman Parung: 536 missing day(s) (e.g. 01 Jan 2024, 02 Jan 2024, 03 Jan 2024, 04 Jan 2024, 05 Jan 2024, ...)
  - KY067 - Kebuli Yaman Metland: 536 missing day(s) (e.g. 01 Jan 2024, 02 Jan 2024, 03 Jan 2024, 04 Jan 2024, 05 Jan 2024, ...)
  - KY065 - Kebuli Yaman Sangiang: 529 missing day(s) (e.g. 01 Jan 2024, 02 Jan 2024, 03 Jan 2024, 04 Jan 2024, 05 Jan 2024, ...)
  - KY064 - Kebuli Yaman Mutiara Garuda: 487 missing day(s) (e.g. 01 Jan 2024, 02 Jan 2024, 03 Jan 2024, 04 Jan 2024, 05 Jan 2024, ...)
  - KY063 - Kebuli Yaman Kedaung: 480 missing day(s) (e.g. 01 Jan 2024, 02 Jan 2024, 03 Jan 2024, 04 Jan 2024, 05 Jan 2024, ...)
  - KY061 - Kebuli Yaman Taman Kirana: 424 missing day(s) (e.g. 01 Jan 2024, 02 Jan 2024, 03 Jan 2024, 04 Jan 2024, 05 Jan 2024, ...)
  - KY062 - Kebuli Yaman Kampung Baru: 424 missing day(s) (e.g. 01 Jan 2024, 02 Jan 2024, 03 Jan 2024, 04 Jan 2024, 05 Jan 2024, ...)
  - KY060 - Kebuli 

In [31]:
# train, test = prepare_forecast_data.split_train_test(featured) # Memotong data menjadi data Train (sebelum 1 Des 2025) dan data Test (selama bulan Des 2025)
# prepare_forecast_data.export_splits(train, test) # Menyimpan data Train dan Test ke dalam file Parquet (train.parquet & test.parquet)
# print(f"Exported {len(train)} train rows and {len(test)} test rows to {prepare_forecast_data.MODEL_READY_DIR}")